In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.3 MB/s eta 0:00:00


In [2]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.5/112.6 GB disk)


In [4]:
import os
from ultralytics import YOLO

def enregistrer_detection_video(model, video_path, output_dir="./predictions_videos"):
    """
    Détecte les personnes dans une vidéo en utilisant un modèle YOLO
    et enregistre la vidéo résultante avec les boîtes englobantes dessinées.

    Args:
        model (ultralytics.YOLO): Le modèle YOLO chargé.
        video_path (str): Le chemin d'accès à la vidéo à analyser.
        output_dir (str): Le répertoire où sauvegarder la vidéo de prédiction.
    """
    os.makedirs(output_dir, exist_ok=True)

    print(f"Détection des personnes dans la vidéo et enregistrement: {video_path}")
    print(f"La vidéo de résultat sera sauvegardée dans: {output_dir}")

    # Class ID for 'person' in COCO dataset (common for YOLO models)
    person_class_id = 0  # Assuming COCO dataset, where 'person' is class 0

    # Run inference on the video and save results directly
    # save=True saves the annotated video to output_dir
    # conf=0.25 (default) is confidence threshold
    # classes=[person_class_id] filters detections to only include 'person'
    model.predict(
        source=video_path,
        save=True,
        project=output_dir,
        name=os.path.basename(video_path).split('.')[0],
        exist_ok=True, # Overwrite existing runs
        classes=[person_class_id]
    )

    print(f"Processus terminé. La vidéo des détections de personnes est enregistrée dans {output_dir}")


model = YOLO("yolo11n.pt")
video_path = "/content/video_personne.mp4"
enregistrer_detection_video(model, video_path)

Détection des personnes dans la vidéo et enregistrement: /content/video_personne.mp4
La vidéo de résultat sera sauvegardée dans: ./predictions_videos

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/300) /content/video_personne.mp4: 384x640 1 person, 50.3ms
video 1/1 (frame 2/300) /content/video_personne.mp4: 384x640 1 person, 9.0ms
video 1/1 (frame 3/300) /content/video_personne.mp4: 384x640 1 person, 8.5ms
video 1/1 (frame 4/300) /content/video_personne.mp4: 384x640 1 person, 7.5ms

In [6]:
import os
from ultralytics import YOLO

def detecter_personne(model, video_path, output_dir="./predictions"):
    """
    Détecte les personnes dans une vidéo en utilisant un modèle YOLO,
    extrait les coordonnées des boîtes englobantes (x, y, w, h) et les
    écrit dans un fichier texte.

    Args:
        model (ultralytics.YOLO): Le modèle YOLO chargé.
        video_path (str): Le chemin d'accès à la vidéo à analyser.
        output_dir (str): Le répertoire où sauvegarder les fichiers de prédiction.
    """
    os.makedirs(output_dir, exist_ok=True)
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    output_file_path = os.path.join(output_dir, f"{video_name}_person_boxes.txt")

    # Class ID for 'person' in COCO dataset (common for YOLO models)
    person_class_id = 0  # Assuming COCO dataset, where 'person' is class 0

    print(f"Détection des personnes dans la vidéo: {video_path}")
    print(f"Les résultats seront sauvegardés dans: {output_file_path}")

    with open(output_file_path, "w") as f:
        # Run inference on the video
        results = model.predict(source=video_path, stream=True)  # stream=True for video/generator

        for frame_idx, result in enumerate(results):
            # Get boxes for the current frame
            boxes = result.boxes

            person_detections_in_frame = []
            for i in range(len(boxes.cls)):
                class_id = int(boxes.cls[i])

                # Check if the detected object is a 'person'
                if class_id == person_class_id:
                    # Extract bounding box coordinates in xywh format
                    # xywh gives center_x, center_y, width, height
                    xywh = boxes.xywh[i].cpu().numpy().flatten()
                    x, y, w, h = xywh[0], xywh[1], xywh[2], xywh[3]
                    person_detections_in_frame.append(f"{x:.2f},{y:.2f},{w:.2f},{h:.2f}")
                    break

            if person_detections_in_frame:
                f.write(f"Frame {frame_idx}: {';'.join(person_detections_in_frame)}\n")

    print(f"Processus terminé. Les boîtes des personnes sont enregistrées dans {output_file_path}")

model = YOLO("yolo11n.pt")
video_path = "/content/video_personne.mp4"
detecter_personne(model, video_path)

Détection des personnes dans la vidéo: /content/video_personne.mp4
Les résultats seront sauvegardés dans: ./predictions/video_personne_person_boxes.txt

video 1/1 (frame 1/300) /content/video_personne.mp4: 384x640 1 person, 18 cars, 9.7ms
video 1/1 (frame 2/300) /content/video_personne.mp4: 384x640 1 person, 18 cars, 9.2ms
video 1/1 (frame 3/300) /content/video_personne.mp4: 384x640 1 person, 18 cars, 7.0ms
video 1/1 (frame 4/300) /content/video_personne.mp4: 384x640 1 person, 18 cars, 7.7ms
video 1/1 (frame 5/300) /content/video_personne.mp4: 384x640 1 person, 18 cars, 7.3ms
video 1/1 (frame 6/300) /content/video_personne.mp4: 384x640 1 person, 18 cars, 7.5ms
video 1/1 (frame 7/300) /content/video_personne.mp4: 384x640 1 person, 18 cars, 7.6ms
video 1/1 (frame 8/300) /content/video_personne.mp4: 384x640 1 person, 18 cars, 11.3ms
video 1/1 (frame 9/300) /content/video_personne.mp4: 384x640 1 person, 18 cars, 8.8ms
video 1/1 (frame 10/300) /content/video_personne.mp4: 384x640 1 person, 